In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ValueError: Mountpoint must not already contain files

In [1]:
from google.colab import drive
drive.flush_and_unmount('/content/drive')
print('Google Drive desmontado.')

Drive not mounted, so nothing to flush and unmount.
Google Drive desmontado.


In [2]:
import os
if os.path.exists('/content/drive'):
    os.system('rm -rf /content/drive')
    print('Diretório /content/drive removido.')
else:
    print('Diretório /content/drive não existe.')

Diretório /content/drive removido.


In [1]:
# ============================================================
#  🔧 CORREÇÃO DE CAMINHOS — Cole numa célula ANTES do treino
#  Você moveu as pastas para dentro de RETINOPATIA/
#  Esse código atualiza os caminhos e recria o que falta
# ============================================================

import os, torch

# ── Caminhos novos (tudo dentro de RETINOPATIA) ──────────────
BASE_PATH        = '/content/drive/MyDrive/RETINOPATIA'
CHECKPOINT_DIR   = '/content/drive/MyDrive/RETINOPATIA/checkpoints_retina'
CHECKPOINT_PATH  = os.path.join(CHECKPOINT_DIR, 'b4_checkpoint.pth')
FINAL_MODEL_PATH = '/content/drive/MyDrive/RETINOPATIA/retinopatia_b4_final.pth'

# ── Cria a pasta de checkpoints se não existir ───────────────
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Verifica o que já existe no Drive ───────────────────────
print('📂 Estrutura atual de RETINOPATIA/:')
for item in sorted(os.listdir(BASE_PATH)):
    full = os.path.join(BASE_PATH, item)
    if os.path.isdir(full):
        qtd = len(os.listdir(full))
        print(f'   📁 {item}/  ({qtd} arquivos)')
    else:
        size_mb = os.path.getsize(full) / 1e6
        print(f'   📄 {item}  ({size_mb:.1f} MB)')

# ── Verifica checkpoint existente ────────────────────────────
# O treino chegou até batch 300 antes de crashar
# Se o checkpoint foi salvo, ele está aqui:
ckpt_antigo = '/content/drive/MyDrive/RETINOPATIA/checkpoints_retina/b4_checkpoint.pth'
if os.path.exists(ckpt_antigo):
    ck = torch.load(ckpt_antigo, map_location='cpu')
    print(f'\n✅ Checkpoint encontrado!')
    print(f'   Epoch: {ck.get("epoch", "?")} | Batch: {ck.get("batch", "?")}')
    print(f'   Melhor val_loss: {ck.get("best_val_loss", "?")}')
else:
    print(f'\n⚠️  Nenhum checkpoint encontrado — vai treinar do batch 0')
    print(f'   (O checkpoint do batch 300 pode ter sido perdido no crash)')

print(f'\n✅ Caminhos atualizados:')
print(f'   BASE_PATH:        {BASE_PATH}')
print(f'   CHECKPOINT_DIR:   {CHECKPOINT_DIR}')
print(f'   CHECKPOINT_PATH:  {CHECKPOINT_PATH}')
print(f'   FINAL_MODEL_PATH: {FINAL_MODEL_PATH}')
print(f'\n👉 Agora cole e rode o TREINO_V2_B4.py na próxima célula')
print(f'   Ele vai retomar do checkpoint se existir, ou começar do zero.')

📂 Estrutura atual de RETINOPATIA/:
   📁 checkpoints_retina/  (0 arquivos)

⚠️  Nenhum checkpoint encontrado — vai treinar do batch 0
   (O checkpoint do batch 300 pode ter sido perdido no crash)

✅ Caminhos atualizados:
   BASE_PATH:        /content/drive/MyDrive/RETINOPATIA
   CHECKPOINT_DIR:   /content/drive/MyDrive/RETINOPATIA/checkpoints_retina
   CHECKPOINT_PATH:  /content/drive/MyDrive/RETINOPATIA/checkpoints_retina/b4_checkpoint.pth
   FINAL_MODEL_PATH: /content/drive/MyDrive/RETINOPATIA/retinopatia_b4_final.pth

👉 Agora cole e rode o TREINO_V2_B4.py na próxima célula
   Ele vai retomar do checkpoint se existir, ou começar do zero.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
#  🧠 TREINO V2 — EfficientNet-B4 | Retinopatia Diabética
#
#  MELHORIAS em relação ao V1:
#  ✅ EfficientNet-B4 (mais preciso que B0)
#  ✅ IMG_SIZE 380 (resolução nativa do B4)
#  ✅ Freeze nas primeiras epochs (treino mais estável)
#  ✅ LR menor (5e-5) → convergência mais suave
#  ✅ Dropout 0.4 na camada final → menos oscilação
#  ✅ 20 epochs (modelo ainda estava aprendendo no V1)
#  ✅ Warmup de LR nas primeiras 2 epochs
#  ✅ Retoma automaticamente do checkpoint do V1 se existir
#  ✅ CLAHE | AMP | Class Weight | Checkpoint frequente
#
#  👉 Cole tudo em UMA célula do Colab e rode
# ============================================================
# ⚡ ANTES DE RODAR: Ative a GPU
#    Ambiente de execução → Alterar tipo de execução → GPU (T4)
# ============================================================

# ── 1. Instalar dependências ──────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "-q", "timm", "albumentations",
                "opencv-python", "scikit-learn", "tqdm"])

# ── 2. Montar Drive ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── 3. Imports ────────────────────────────────────────────────
import os, gc, time, warnings
import torch
import torch.nn as nn
import timm
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
import torch.optim as optim

import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')

# ── 4. Verificar GPU ──────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  GPU não detectada! Vá em: Ambiente de execução → Alterar tipo → GPU')

# ── 5. Configurações ──────────────────────────────────────────
BASE_PATH            = '/content/drive/MyDrive/RETINOPATIA'
CHECKPOINT_DIR       = '/content/drive/MyDrive/checkpoints_retina'
CHECKPOINT_DIR   = '/content/drive/MyDrive/RETINOPATIA/checkpoints_retina'
FINAL_MODEL_PATH = '/content/drive/MyDrive/RETINOPATIA/retinopatia_b4_final.pth'  # novo arquivo
CHECKPOINT_PATH  = os.path.join(CHECKPOINT_DIR, 'b4_checkpoint.pth') # Adicionada esta linha
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

IMG_SIZE             = 380    # resolução nativa do EfficientNet-B4
BATCH_SIZE           = 16     # B4 é maior — batch menor pra caber na GPU
NUM_WORKERS          = 4
EPOCHS               = 20     # mais epochs para convergir melhor
LR                   = 5e-5   # LR menor = convergência mais suave
FREEZE_EPOCHS        = 3      # primeiras N epochs: só treina a cabeça (mais estável)
SAVE_EVERY_N_BATCHES = 300
EARLY_STOP_PATIENCE  = 5      # mais paciência que o V1
USE_AMP              = True

CLASSES = {
    'No_DR': 0, 'Mild': 1, 'Moderate': 2,
    'Severe': 3, 'Proliferate_DR': 4
}

print(f'\n✅ Config V2: IMG={IMG_SIZE} | BATCH={BATCH_SIZE} | EPOCHS={EPOCHS}')
print(f'   LR={LR} | FREEZE={FREEZE_EPOCHS} epochs | AMP={USE_AMP}')

# ── 6. Montar DataFrame ───────────────────────────────────────
data = []
for c in CLASSES:
    folder = os.path.join(BASE_PATH, c)
    for img in os.listdir(folder):
        data.append([os.path.join(folder, img), CLASSES[c]])

df = pd.DataFrame(data, columns=['image', 'label'])
print(f'\nTotal de imagens: {len(df)}')
print(df['label'].value_counts().sort_index())

# ── 7. Split ──────────────────────────────────────────────────
train_df, val_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)
print(f'\nTreino: {len(train_df)} | Validação: {len(val_df)}')

# ── 8. CLAHE ──────────────────────────────────────────────────
def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_clahe, a, b]), cv2.COLOR_LAB2RGB)

# ── 9. Transformações ─────────────────────────────────────────
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.95, 1.05), translate_percent=0.05, rotate=(-180, 180), p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ── 10. Dataset ───────────────────────────────────────────────
class RetinaDataset(Dataset):
    def __init__(self, df, transform=None, use_clahe=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.use_clahe = use_clahe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.image)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.use_clahe:
            img = apply_clahe(img)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, int(row.label)

# ── 11. DataLoaders ───────────────────────────────────────────
train_dataset = RetinaDataset(train_df, train_transform, use_clahe=True)
val_dataset   = RetinaDataset(val_df,   val_transform,   use_clahe=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device=='cuda'),
                          persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device=='cuda'),
                          persistent_workers=(NUM_WORKERS > 0))

print(f'\nBatches treino: {len(train_loader)} | Batches val: {len(val_loader)}')

# ── 12. Modelo B4 com Dropout ─────────────────────────────────
model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=0)
# Dropout 0.4 antes da camada final — reduz oscilação da acurácia
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(model.num_features, 5)
)
model = model.to(device)

def freeze_backbone(m):
    """Congela tudo exceto a cabeça (classifier)."""
    for name, param in m.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False

def unfreeze_all(m):
    """Descongela todas as camadas para fine-tuning completo."""
    for param in m.parameters():
        param.requires_grad = True

# Começa com backbone congelado (só treina a cabeça nas primeiras epochs)
freeze_backbone(model)
print('✅ Modelo B4 carregado — backbone congelado para as primeiras epochs')

try:
    model = torch.compile(model)
    print('✅ torch.compile() ativado')
except Exception:
    print('ℹ️  torch.compile() indisponível — ok')

# ── 13. Class Weight ──────────────────────────────────────────
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2, 3, 4]),
    y=train_df['label'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print('\n📊 Distribuição e pesos:')
print(f'{"Classe":<20} {"Qtd":>8} {"Peso":>8}')
print('-' * 40)
nomes = ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferate_DR']
contagens = train_df['label'].value_counts().sort_index()
for idx, (nome, peso) in enumerate(zip(nomes, class_weights)):
    print(f'{nome:<20} {contagens.get(idx,0):>8} {peso:>8.4f}')

# ── 14. Loss / Optimizer / Scaler ────────────────────────────
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)
scaler = GradScaler(enabled=(USE_AMP and device == 'cuda'))

print('\n✅ Loss, optimizer, scheduler e scaler prontos')

# ── 15. Carregar checkpoint se existir ───────────────────────
start_epoch        = 0
start_batch        = 0
best_val_loss      = float('inf')
early_stop_counter = 0
history            = []

if os.path.exists(CHECKPOINT_PATH):
    print('\n📂 Checkpoint B4 encontrado — retomando...')
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch        = ckpt.get('epoch', 0)
    start_batch        = ckpt.get('batch', 0)
    best_val_loss      = ckpt.get('best_val_loss', float('inf'))
    early_stop_counter = ckpt.get('early_stop_counter', 0)
    history            = ckpt.get('history', [])
    if start_batch >= len(train_loader):
        start_epoch += 1
        start_batch  = 0
    print(f'   Retomando epoch {start_epoch+1} | best_val_loss {best_val_loss:.4f}')
else:
    print('\n🆕 Nenhum checkpoint B4 — treinando do zero')

# ── 16. Funções auxiliares ────────────────────────────────────
def save_checkpoint(epoch, batch, tag=''):
    torch.save({
        'epoch': epoch, 'batch': batch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict(),
        'best_val_loss': best_val_loss,
        'early_stop_counter': early_stop_counter,
        'history': history,
    }, CHECKPOINT_PATH)
    if tag:
        print(f'   💾 Checkpoint salvo {tag}')

def validate():
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc='Validação', leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast(enabled=(USE_AMP and device == 'cuda')):
                preds = model(images)
                loss  = criterion(preds, labels)
            total_loss += loss.item()
            correct    += (preds.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(val_loader), correct / total

# ── 17. Loop de treinamento ───────────────────────────────────
print(f'\n🚀 Treinando B4 — epochs {start_epoch+1} → {EPOCHS}')
print(f'   Epochs 1-{FREEZE_EPOCHS}: backbone congelado (só treina cabeça)')
print(f'   Epochs {FREEZE_EPOCHS+1}+: fine-tuning completo')
print('=' * 60)

for epoch in range(start_epoch, EPOCHS):

    # ── Descongelar backbone após FREEZE_EPOCHS ───────────────
    if epoch == FREEZE_EPOCHS:
        unfreeze_all(model)
        # Recria optimizer com LR menor para fine-tuning completo
        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=LR / 2)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3
        )
        print(f'\n🔓 Backbone descongelado! Fine-tuning completo com LR={LR/2:.1e}')

    model.train()
    total_loss  = 0.0
    epoch_start = time.time()

    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f'Epoch {epoch+1}/{EPOCHS}', dynamic_ncols=True)

    for batch_idx, (images, labels) in pbar:

        if epoch == start_epoch and batch_idx < start_batch:
            continue

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast(enabled=(USE_AMP and device == 'cuda')):
            preds = model(images)
            loss  = criterion(preds, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        avg_loss    = total_loss / (batch_idx - start_batch + 1)

        pbar.set_postfix({'loss': f'{avg_loss:.4f}',
                          'lr':   f'{optimizer.param_groups[0]["lr"]:.2e}'})

        if (batch_idx + 1) % SAVE_EVERY_N_BATCHES == 0:
            save_checkpoint(epoch, batch_idx + 1,
                            tag=f'(epoch {epoch+1}, batch {batch_idx+1})')

    start_batch = 0

    val_loss, val_acc = validate()
    elapsed = time.time() - epoch_start

    status = '🔒 frozen' if epoch < FREEZE_EPOCHS else '🔓 full'
    print(f'\nEpoch {epoch+1}/{EPOCHS} [{status}] | '
          f'Train Loss: {avg_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'Val Acc: {val_acc:.4f} | '
          f'Tempo: {elapsed/60:.1f} min')

    history.append({'epoch': epoch+1, 'train_loss': avg_loss,
                    'val_loss': val_loss, 'val_acc': val_acc,
                    'frozen': epoch < FREEZE_EPOCHS})

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss      = val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), FINAL_MODEL_PATH)
        print(f'   ⭐ Melhor modelo salvo! Val Loss: {val_loss:.4f}')
    else:
        early_stop_counter += 1
        print(f'   ⚠️  Sem melhoria ({early_stop_counter}/{EARLY_STOP_PATIENCE})')
        if early_stop_counter >= EARLY_STOP_PATIENCE:
            print('🛑 Early stopping!')
            save_checkpoint(epoch, len(train_loader), tag='(early stop)')
            break

    save_checkpoint(epoch + 1, 0, tag=f'(fim epoch {epoch+1})')
    print('=' * 60)

print('\n🏁 Treino V2 finalizado!')

# ── 18. Gráficos ─────────────────────────────────────────────
if history:
    epochs_done  = [h['epoch']      for h in history]
    train_losses = [h['train_loss'] for h in history]
    val_losses   = [h['val_loss']   for h in history]
    val_accs     = [h['val_acc']    for h in history]

    # Marca onde o freeze terminou
    freeze_end = min(FREEZE_EPOCHS, len(epochs_done))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs_done, train_losses, label='Train Loss', marker='o')
    ax1.plot(epochs_done, val_losses,   label='Val Loss',   marker='o')
    if freeze_end < len(epochs_done):
        ax1.axvline(x=freeze_end + 0.5, color='gray', linestyle='--',
                    label=f'Unfreeze (epoch {freeze_end+1})')
    ax1.set_title('Loss por Epoch — B4'); ax1.legend(); ax1.grid(True)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')

    ax2.plot(epochs_done, val_accs, label='Val Accuracy', color='green', marker='o')
    if freeze_end < len(epochs_done):
        ax2.axvline(x=freeze_end + 0.5, color='gray', linestyle='--',
                    label=f'Unfreeze (epoch {freeze_end+1})')
    ax2.set_title('Acurácia de Validação — B4'); ax2.legend(); ax2.grid(True)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')

    plt.suptitle('EfficientNet-B4 | Retinopatia Diabética', fontsize=13)
    plt.tight_layout()
    plt.show()

    print(pd.DataFrame(history)[['epoch','train_loss','val_loss','val_acc']].to_string(index=False))

# ── 19. Verificar arquivos ────────────────────────────────────
print('\n📁 Arquivos salvos:')
for path, label in [(CHECKPOINT_PATH, 'Checkpoint B4'),
                    (FINAL_MODEL_PATH, 'Modelo B4 final')]:
    if os.path.exists(path):
        print(f'  ✅ {label}: {path} ({os.path.getsize(path)/1e6:.1f} MB)')
    else:
        print(f'  ❌ {label} não encontrado: {path}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo: cpu
⚠️  GPU não detectada! Vá em: Ambiente de execução → Alterar tipo → GPU

✅ Config V2: IMG=380 | BATCH=16 | EPOCHS=20
   LR=5e-05 | FREEZE=3 epochs | AMP=True

Total de imagens: 35135
label
0    25817
1     2444
2     5292
3      873
4      709
Name: count, dtype: int64

Treino: 28108 | Validação: 7027

Batches treino: 1757 | Batches val: 440
✅ Modelo B4 carregado — backbone congelado para as primeiras epochs
✅ torch.compile() ativado

📊 Distribuição e pesos:
Classe                    Qtd     Peso
----------------------------------------
No_DR                   20654   0.2722
Mild                     1955   2.8755
Moderate                 4234   1.3277
Severe                    698   8.0539
Proliferate_DR            567   9.9146

✅ Loss, optimizer, scheduler e scaler prontos

📂 Checkpoint B4 encontrado — retomando...
   Retomando epoch 1 | bes

Epoch 1/20:   0%|          | 0/1757 [00:00<?, ?it/s]